**Week 13 — GLM (Poisson regression), ROC/AUC, class imbalance และการเปรียบเทียบ classifier (ข้อมูลลูกค้าร้านค้า)**

### สรุปเงื่อนไขการประมวลผลข้อมูล
- **การอ่านไฟล์:** `cust = pd.read_csv('biz_customers.csv')` (1203 แถว แถวละ 1 คน)
- **การแบ่งชุดข้อมูล:** train = 960 แถวแรก (`.iloc[:960]`), test = 243 แถวที่เหลือ (`.iloc[960:]`) โดยไม่สลับข้อมูล
- **Poisson Regression:** ใช้ `smf.glm(..., family=sm.families.Poisson())` (Log link) เสมอ โดย $\text{IRR} = e^{\beta}$
- **Logistic Regression:** ประมาณค่าด้วย Maximum Likelihood Estimation (`smf.logit` ไม่มี Regularization)
- **การจำแนกประเภท:** คลาสบวก = 1 และความน่าจะเป็นที่ใช้ประเมินคือ $P(Y=1)$

### รายละเอียดไฟล์ข้อมูล `biz_customers.csv` (1203 แถว)

- `age`: อายุของลูกค้า (ปี)
- `income_k`: รายได้ของลูกค้า (พันบาท/เดือน)
- `tenure_months`: ระยะเวลาที่เป็นลูกค้า (เดือน)
- `loyalty_member`: การเป็นสมาชิกสะสมแต้ม (1 = เป็น, 0 = ไม่เป็น)
- `web_visits`: จำนวนครั้งที่เข้าเว็บ/แอปใน 30 วันล่าสุด
- `email_opens`: จำนวนอีเมลโปรโมชันที่เปิดอ่าน (จาก 20 ฉบับ)
- `basket_size`: จำนวนสินค้าเฉลี่ยต่อออเดอร์
- `orders_90d`: จำนวนออเดอร์ใน 90 วันล่าสุด
- `churn`: การเลิกซื้อ (1 = เลิกซื้อ, 0 = ยังซื้ออยู่)
- `responded`: การตอบรับโปรโมชัน (1 = ตอบรับ, 0 = ไม่ตอบรับ)

**โจทย์ข้อ 13.1:** ฝึกใช้ `biz_customers.csv` ลูกค้าทั้ง 1203 คน ทำ Poisson regression (log link, `smf.glm` ร่วมกับ `sm.families.Poisson()`) โดย $y$ = `orders_90d` ตัวทำนาย = `loyalty_member`, `tenure_months`, `income_k` ค่า **IRR ($= e^{\text{สัมประสิทธิ์}}$) ของ `loyalty_member`** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1. อ่านข้อมูลลูกค้าทั้งหมด
cust = pd.read_csv('biz_customers.csv')

# 2. Fit Poisson Regression
formula = 'orders_90d ~ loyalty_member + tenure_months + income_k'
model = smf.glm(formula, data=cust, family=sm.families.Poisson()).fit()
print(model.params)

# 3. คำนวณ IRR = exp(beta) ของ loyalty_member
coef_loyalty = model.params['loyalty_member']
irr_loyalty = np.exp(coef_loyalty)
print(f"สัมประสิทธิ์ loyalty_member: {coef_loyalty:.4f}")
print(f"Incidence Rate Ratio (IRR): {irr_loyalty:.4f}")
print(f"คำตอบข้อ 13.1: {irr_loyalty:.4f}")


Intercept        -0.026263
loyalty_member    0.483647
tenure_months     0.004106
income_k          0.006835
dtype: float64
สัมประสิทธิ์ loyalty_member: 0.4836
Incidence Rate Ratio (IRR): 1.6220
คำตอบข้อ 13.1: 1.6220


**โจทย์ข้อ 13.2:** ฝึกใช้ LDA (`LinearDiscriminantAnalysis` ค่าเริ่มต้น ไม่ต้อง standardize) ทำนาย `responded` จาก `web_visits`, `basket_size`, `income_k` ในไฟล์ `biz_customers.csv` fit บน train (960 แถวแรกของไฟล์) แล้วหาค่า **AUC ของชุด test** (แถวที่เหลือ 243 แถว) โดยใช้ความน่าจะเป็นของคลาส 1 = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import roc_auc_score

# 1. แบ่งชุดข้อมูล train / test
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]
features = ['web_visits', 'basket_size', 'income_k']

# 2. Fit LDA บน train
lda = LinearDiscriminantAnalysis().fit(train[features], train['responded'])

# 3. ทำนายความน่าจะเป็นของคลาส 1 บน test และหาค่า AUC
prob_test = lda.predict_proba(test[features])[:, 1]
auc = roc_auc_score(test['responded'], prob_test)
print(f"AUC บนชุด test: {auc:.4f}")
print(f"คำตอบข้อ 13.2: {auc:.4f}")


AUC บนชุด test: 0.6776
คำตอบข้อ 13.2: 0.6776


**โจทย์ข้อ 13.3:** กรณีศึกษาศูนย์กระจายสินค้าของร้านค้าออนไลน์ต้องประมาณปริมาณออเดอร์ล่วงหน้าสำหรับลูกค้ารายใหม่ที่ลงทะเบียนเข้ามา ทีมใช้ลูกค้าทั้ง 1203 คนใน `biz_customers.csv` ทำ Poisson regression (log link) ทำนายจำนวนออเดอร์ใน 90 วันล่าสุด จาก จำนวนครั้งที่เข้าเว็บ/แอปใน 30 วันล่าสุด, การเป็นสมาชิกสะสมแต้ม (1 = เป็น), ระยะเวลาที่เป็นลูกค้า (เดือน), รายได้ของลูกค้า (พันบาท/เดือน) ลูกค้ารายหนึ่งเข้าเว็บ/แอป 4 ครั้งใน 30 วัน, เป็นสมาชิกสะสมแต้ม, เป็นลูกค้ามา 44 เดือน, รายได้ 57 พันบาท/เดือน **จำนวนออเดอร์ใน 90 วันที่โมเดลคาดว่าลูกค้ารายนี้จะสั่งโดยเฉลี่ย** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1. Fit Poisson Regression บนลูกค้าทุกคน
cust = pd.read_csv('biz_customers.csv')
formula = 'orders_90d ~ web_visits + loyalty_member + tenure_months + income_k'
model = smf.glm(formula, data=cust, family=sm.families.Poisson()).fit()

# 2. กำหนดข้อมูลลูกค้ารายใหม่
new_customer = pd.DataFrame([{
    'web_visits': 4,
    'loyalty_member': 1,
    'tenure_months': 44,
    'income_k': 57
}])

# 3. พยากรณ์ค่าคาดหวังจำนวนออเดอร์
pred_orders = model.predict(new_customer)[0]
print(f"จำนวนออเดอร์เฉลี่ยที่คาดการณ์ได้: {pred_orders:.4f}")
print(f"คำตอบข้อ 13.3: {pred_orders:.4f}")


จำนวนออเดอร์เฉลี่ยที่คาดการณ์ได้: 1.8004
คำตอบข้อ 13.3: 1.8004


**โจทย์ข้อ 13.4:** ฝึกทำ Poisson regression (log link) $y$ = `orders_90d` ตัวทำนาย = `web_visits`, `loyalty_member`, `tenure_months` ด้วยลูกค้าทั้ง 1203 คนใน `biz_customers.csv` **ค่าสถิติ $z$ (สัมประสิทธิ์ $\div$ standard error ตามที่ `summary()` ของ smf.glm Poisson รายงาน) ของ `tenure_months`** = ? (ทศนิยม 3 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1. Fit Poisson Regression
cust = pd.read_csv('biz_customers.csv')
formula = 'orders_90d ~ web_visits + loyalty_member + tenure_months'
model = smf.glm(formula, data=cust, family=sm.families.Poisson()).fit()
print(model.summary())

# 2. ดึงค่าสถิติ z ของ tenure_months
z_tenure = model.tvalues['tenure_months']
print(f"z-statistic ของ tenure_months: {z_tenure:.3f}")
print(f"คำตอบข้อ 13.4: {z_tenure:.3f}")


z-statistic ของ tenure_months: 3.644
คำตอบข้อ 13.4: 3.644


**โจทย์ข้อ 13.5:** กรณีศึกษาฝ่ายวางแผนของแพลตฟอร์มอีคอมเมิร์ซอยากรู้ว่าลูกค้าสองคนที่ลักษณะเหมือนกันทุกอย่าง แต่คนหนึ่งมีระยะเวลาที่เป็นลูกค้ามากกว่าอีกคน 24 เดือน จะสั่งซื้อต่างกันกี่เท่า ใช้ลูกค้าทั้ง 1203 คนใน `biz_customers.csv` ทำ Poisson regression (log link) ทำนายจำนวนออเดอร์ใน 90 วันล่าสุด จาก จำนวนครั้งที่เข้าเว็บ/แอปใน 30 วันล่าสุด, การเป็นสมาชิกสะสมแต้ม (1 = เป็น), ระยะเวลาที่เป็นลูกค้า (เดือน), รายได้ของลูกค้า (พันบาท/เดือน) (ไม่มี interaction) **จำนวนออเดอร์เฉลี่ยของคนที่มากกว่าเป็นกี่เท่าของอีกคน (ตัวแปรอื่นเท่ากัน)** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1. Fit Poisson Regression
cust = pd.read_csv('biz_customers.csv')
formula = 'orders_90d ~ web_visits + loyalty_member + tenure_months + income_k'
model = smf.glm(formula, data=cust, family=sm.families.Poisson()).fit()

# 2. คำนวณอัตราส่วนเมื่อ tenure_months เพิ่มขึ้น 24 เดือน = exp(24 * beta)
beta_tenure = model.params['tenure_months']
ratio_24m = np.exp(24.0 * beta_tenure)
print(f"beta ของ tenure_months: {beta_tenure:.6f}")
print(f"อัตราส่วนการสั่งซื้อ (exp(24 * beta)): {ratio_24m:.4f}")
print(f"คำตอบข้อ 13.5: {ratio_24m:.4f}")


beta ของ tenure_months: 0.004036
อัตราส่วนการสั่งซื้อ (exp(24 * beta)): 1.1017
คำตอบข้อ 13.5: 1.1017


**โจทย์ข้อ 13.6:** กรณีศึกษาบริษัทประกันพบว่าลูกค้าที่เลิกต่ออายุ (เลิกซื้อ) มีเพียงส่วนน้อย จึงจะ fit classifier ด้วย `class_weight='balanced'` ซึ่งให้น้ำหนักคลาส $c$ เท่ากับ $n_{\text{samples}} \div (n_{\text{classes}} \times \text{จำนวนตัวอย่างของคลาส } c)$ ใช้ train (960 แถวแรกของ `biz_customers.csv`) ที่มีคอลัมน์ `churn` (1 = เลิกซื้อ) เป็นตัวแปรที่จะทำนาย **น้ำหนักที่คลาส `churn = 1` ได้รับ** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านชุด train (960 แถวแรก)
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]

# 2. คำนวณน้ำหนัก balanced สำหรับ churn = 1
n_samples = len(train)
n_classes = 2
n_c1 = int(np.sum(train['churn'] == 1))
weight_c1 = n_samples / (n_classes * n_c1)

print(f"จำนวนตัวอย่างทั้งหมด: {n_samples}")
print(f"จำนวนตัวอย่าง churn = 1: {n_c1}")
print(f"น้ำหนักของคลาส 1: {weight_c1:.4f}")
print(f"คำตอบข้อ 13.6: {weight_c1:.4f}")


จำนวนตัวอย่างทั้งหมด: 960
จำนวนตัวอย่าง churn = 1: 115
น้ำหนักของคลาส 1: 4.1739
คำตอบข้อ 13.6: 4.1739


**โจทย์ข้อ 13.7:** กรณีศึกษาทีมการตลาดของธนาคารจะส่งข้อเสนอบัตรเครดิตโดยไม่ยอมให้พลาดลูกค้าที่จะตอบรับจริง (ต้องการสัดส่วนผู้ตอบรับที่จับได้ หรือ $\text{TPR} \ge 0.85$) fit Logistic regression (ประมาณด้วย Maximum Likelihood ไม่มี regularization) ทำนายการตอบรับโปรโมชัน (`responded`) จาก อายุของลูกค้า (ปี), จำนวนอีเมลโปรโมชันที่เปิดอ่าน (จาก 20 ฉบับ), รายได้ของลูกค้า (พันบาท/เดือน) บน train (960 แถวแรกของ `biz_customers.csv`) แล้วใช้ test สร้าง ROC curve (ทำนายเป็น 1 เมื่อความน่าจะเป็น $\ge$ threshold) จาก threshold ที่ `roc_curve` ส่งคืน เลือก threshold ที่สูงที่สุดที่ยังทำให้ $\text{TPR} \ge 0.85$ ที่จุดนั้น **สัดส่วนของลูกค้าที่ไม่ตอบรับ (ใน test) ที่ถูกส่งข้อเสนอด้วย (FPR)** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import roc_curve

# 1. แบ่งชุด train / test
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]

# 2. Fit โมเดล Logit บน train
formula = 'responded ~ age + email_opens + income_k'
model = smf.logit(formula, data=train).fit()

# 3. สร้าง ROC Curve บน test
prob_test = model.predict(test)
fpr, tpr, thresholds = roc_curve(test['responded'], prob_test)

# 4. เลือกจุดที่ TPR >= 0.85 โดยมี threshold สูงสุด (จุดแรกที่เงื่อนไขเป็นจริง)
valid_idx = np.where(tpr >= 0.85)[0][0]
best_threshold = thresholds[valid_idx]
best_tpr = tpr[valid_idx]
best_fpr = fpr[valid_idx]

print(f"Threshold ที่เลือก: {best_threshold:.4f}")
print(f"TPR: {best_tpr:.4f}, FPR: {best_fpr:.4f}")
print(f"คำตอบข้อ 13.7: {best_fpr:.4f}")


Optimization terminated successfully.
         Current function value: 0.588269
         Iterations 5
Threshold ที่เลือก: 0.2421
TPR: 0.8608, FPR: 0.5793
คำตอบข้อ 13.7: 0.5793


**โจทย์ข้อ 13.8:** ฝึกเปรียบเทียบ classifier 4 แบบเพื่อทำนาย `responded` จาก `age`, `web_visits`, `tenure_months` fit บน train (960 แถวแรกของ `biz_customers.csv`) ประเมินบน test: (1) Logistic regression (Maximum Likelihood, ไม่มี regularization) (2) LDA (3) QDA (4) Gaussian Naive Bayes (ทุกแบบใช้ค่าตัวแปรดิบ ไม่ต้อง standardize ใช้ความน่าจะเป็นของคลาส 1) **ค่า AUC ของ test ที่สูงที่สุดในทั้ง 4 แบบ** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score

# 1. เตรียมข้อมูล
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]
features = ['age', 'web_visits', 'tenure_months']

# 2. โมเดลที่ 1: Logistic Regression
m_logit = smf.logit('responded ~ age + web_visits + tenure_months', data=train).fit()
auc_logit = roc_auc_score(test['responded'], m_logit.predict(test))

# 3. โมเดลที่ 2: LDA
m_lda = LinearDiscriminantAnalysis().fit(train[features], train['responded'])
auc_lda = roc_auc_score(test['responded'], m_lda.predict_proba(test[features])[:, 1])

# 4. โมเดลที่ 3: QDA
m_qda = QuadraticDiscriminantAnalysis().fit(train[features], train['responded'])
auc_qda = roc_auc_score(test['responded'], m_qda.predict_proba(test[features])[:, 1])

# 5. โมเดลที่ 4: Gaussian Naive Bayes
m_gnb = GaussianNB().fit(train[features], train['responded'])
auc_gnb = roc_auc_score(test['responded'], m_gnb.predict_proba(test[features])[:, 1])

print(f"Logit AUC: {auc_logit:.4f}")
print(f"LDA AUC:   {auc_lda:.4f}")
print(f"QDA AUC:   {auc_qda:.4f}")
print(f"GNB AUC:   {auc_gnb:.4f}")

max_auc = max(auc_logit, auc_lda, auc_qda, auc_gnb)
print(f"ค่า AUC สูงสุด: {max_auc:.4f}")
print(f"คำตอบข้อ 13.8: {max_auc:.4f}")


Optimization terminated successfully.
         Current function value: 0.593065
         Iterations 5
Logit AUC: 0.6889
LDA AUC:   0.6884
QDA AUC:   0.6903
GNB AUC:   0.6933
ค่า AUC สูงสุด: 0.6933
คำตอบข้อ 13.8: 0.6933


**โจทย์ข้อ 13.9:** กรณีศึกษาธุรกิจกล่องสมาชิกรายเดือนพบว่าลูกค้าที่เลิกซื้อมีน้อยมาก ถ้าใช้ threshold 0.5 โมเดลแทบไม่ทำนายว่าจะเลิกซื้อเลย ทีมรักษาลูกค้าจึงลด threshold ให้เท่ากับสัดส่วนผู้เลิกซื้อใน train fit Logistic regression (Maximum Likelihood) ทำนายการเลิกซื้อ (`churn`) จาก ระยะเวลาที่เป็นลูกค้า (เดือน), จำนวนอีเมลโปรโมชันที่เปิดอ่าน (จาก 20 ฉบับ), จำนวนครั้งที่เข้าเว็บ/แอปใน 30 วันล่าสุด บน train (960 แถวแรกของ `biz_customers.csv`) ทำนายเป็น 1 เมื่อความน่าจะเป็น $\ge$ threshold นั้น แล้วหา **recall (สัดส่วนผู้เลิกซื้อจริงที่ทำนายถูก) บน test** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import recall_score

# 1. แบ่งชุด train / test และหา base rate threshold
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]
threshold_churn = train['churn'].mean()
print(f"Threshold ที่ใช้ (สัดส่วน churn ใน train): {threshold_churn:.4f}")

# 2. Fit โมเดล Logit บน train
formula = 'churn ~ tenure_months + email_opens + web_visits'
model = smf.logit(formula, data=train).fit()

# 3. ทำนายบน test ด้วย threshold ที่กำหนด และหาค่า Recall
prob_test = model.predict(test)
pred_test = (prob_test >= threshold_churn).astype(int)
recall = recall_score(test['churn'], pred_test)
print(f"Recall บนชุด test: {recall:.4f}")
print(f"คำตอบข้อ 13.9: {recall:.4f}")


Threshold ที่ใช้ (สัดส่วน churn ใน train): 0.1198
Optimization terminated successfully.
         Current function value: 0.317910
         Iterations 7
Recall บนชุด test: 0.5714
คำตอบข้อ 13.9: 0.5714


**โจทย์ข้อ 13.10:** ฝึกถ้าใช้ค่า `age` ของลูกค้าโดยตรงเป็น "คะแนน" (ค่ามากแปลว่ามีแนวโน้มเป็นคลาส 1 มากกว่า) เพื่อจำแนก `churn` ใช้ลูกค้าทั้ง 1203 คนใน `biz_customers.csv` (ไม่ต้อง fit โมเดล ไม่ต้องแบ่ง train/test) **ค่า AUC** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# 1. อ่านข้อมูลลูกค้าทั้งหมด
cust = pd.read_csv('biz_customers.csv')

# 2. คำนวณ ROC AUC โดยใช้ age เป็นคะแนนจำแนก churn โดยตรง
auc_age = roc_auc_score(cust['churn'], cust['age'])
print(f"AUC ของตัวแปร age ในการจำแนก churn: {auc_age:.4f}")
print(f"คำตอบข้อ 13.10: {auc_age:.4f}")


AUC ของตัวแปร age ในการจำแนก churn: 0.5998
คำตอบข้อ 13.10: 0.5998


## สรุปทฤษฎีและแนวคิดหลักประจำ Week 13

---

### 1. แบบจำลองเชิงเส้นนัยทั่วไป (GLM) และ Poisson Regression

เมื่อตัวแปรตอบสนอง $Y$ เป็นข้อมูลจำนวนนับไม่เป็นลบ ($Y \in \{0, 1, 2, \dots\}$ เช่น จำนวนออเดอร์) สมมติฐานความแปรปรวนคงที่ของการถดถอยเชิงเส้นธรรมดาจะไม่เป็นจริง จึงนิยมใช้ **Poisson Regression** โดยมี Log Link Function:
$$\ln(\lambda) = \beta_0 + \sum_{j=1}^p \beta_j X_j \implies \lambda = \mathbb{E}[Y \mid X] = e^{\beta_0 + \sum \beta_j X_j}$$

- **Incidence Rate Ratio (IRR):**
  $$\text{IRR}_j = e^{\beta_j}$$
  เมื่อ $X_j$ เพิ่มขึ้น 1 หน่วย ค่าเฉลี่ยของจำนวนนับจะเปลี่ยนไปเป็น $e^{\beta_j}$ เท่า
- หากตัวแปรเพิ่มขึ้น $\Delta$ หน่วย อัตราส่วนจะเปลี่ยนไปเป็น $e^{\Delta \cdot \beta_j}$ เท่า

---

### 2. กราฟ ROC และพื้นที่ใต้กราฟ (AUC: Area Under the Curve)

- **ROC Curve:** กราฟความสัมพันธ์ระหว่าง $\text{TPR}$ (Recall) บนแกนตั้ง กับ $\text{FPR}$ ($1 - \text{Specificity}$) บนแกนนอน เมื่อปรับเปลี่ยน Decision Threshold ตั้งแต่ 0 ถึง 1
- **AUC (Area Under the ROC Curve):**
  - เป็นการวัดความสามารถเชิงจัดลำดับ (Ranking ability) ของโมเดล โดย $\text{AUC} = P(\text{Score}(Y=1) > \text{Score}(Y=0))$
  - $\text{AUC} = 0.5$: ไม่ต่างจากการเดาสุ่ม
  - $\text{AUC} = 1.0$: จำแนกคลาสบวกและลบได้อย่างสมบูรณ์แบบ

---

### 3. การจัดการปัญหาข้อมูลไม่สมดุล (Class Imbalance Strategies)

1. **Cost-Sensitive Threshold Tuning:** ในงานธุรกิจที่คลาสบวกมีสัดส่วนน้อย (เช่น Churn มีเพียง ~12%) การใช้เกณฑ์ $0.5$ จะทำให้ทำนายคลาส 1 น้อยเกินไป การปรับลด threshold มาที่ Base Rate (สัดส่วนจริงของคลาสบวกในชุดฝึก) จะช่วยเพิ่ม **Recall** ได้อย่างเหมาะสม
2. **Balanced Sample Weighting:** ให้น้ำหนักตัวอย่างแต่ละคลาสผกผันกับจำนวนตัวอย่างในชุดฝึก:
   $$w_c = \frac{n_{\text{samples}}}{n_{\text{classes}} \times n_c}$$
   เพื่อลงโทษความผิดพลาดในคลาสส่วนน้อยให้หนักขึ้น

---

### 4. การเปรียบเทียบระหว่างโมเดลจำแนกประเภท (Model Comparison)

- **Logistic Regression:** ไม่ต้องมีสมมติฐานเรื่องการแจกแจงของ $X$ ทนทานและตีความง่าย
- **LDA / QDA:** ทำงานได้ดีเมื่อ $X$ มีการแจกแจงใกล้เคียง Multivariate Normal
- **Gaussian Naive Bayes:** แม้จะมีสมมติฐานว่าตัวแปรอิสระต่อกันอย่างเคร่งครัด แต่ในการจัดอันดับคะแนนความน่าจะเป็น (Ranking) เพื่อคำนวณ AUC มักให้ผลลัพธ์ที่ดีเยี่ยมเมื่อข้อมูลมีสัญญาณชัดเจน